# Train Stage 2 YOLO26s Classifier — Vietnam

Notebook này dành cho Google Colab GPU và chỉ dùng YOLO26s classification.

- Model bắt buộc: `yolo26s-cls.pt`.
- Dataset: `Stage2_Vietnam_Clean`.
- Input: 224, epochs 50, batch 32, AdamW.
- Test chỉ được kiểm tra folder và số lượng ảnh; không inference, không `val`, không tune threshold.
- Threshold và metric chỉ được chọn trên CLEAN validation.
- Cell training chưa được chạy khi notebook được tạo.

In [ ]:
%pip install -q ultralytics==8.4.155

import csv
import inspect
import json
import math
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from google.colab import drive
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from ultralytics import YOLO
from ultralytics.data.utils import check_cls_dataset

assert ultralytics.__version__ == '8.4.155', ultralytics.__version__
print('Ultralytics:', ultralytics.__version__)

In [ ]:
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
PREFERRED_DATASET = MYDRIVE / 'Stage2_Vietnam_Clean'
OUTPUT_ROOT = MYDRIVE / 'DetectedHelmets'
PROJECT_DIR = OUTPUT_ROOT / 'runs' / 'classify'
RUN_NAME = 'stage2_yolo26s_cls_vietnam'
RUN_DIR = PROJECT_DIR / RUN_NAME
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = ('train', 'valid', 'test')
CLASS_NAMES = ('helmet', 'no_helmet')
EXPECTED = {
    'train': {'helmet': 852, 'no_helmet': 765},
    'valid': {'helmet': 115, 'no_helmet': 72},
    'test': {'helmet': 107, 'no_helmet': 71},
}

def has_clean_layout(root):
    return all((root / split / class_name).is_dir() for split in SPLITS for class_name in CLASS_NAMES)

def find_clean_dataset():
    if has_clean_layout(PREFERRED_DATASET):
        return PREFERRED_DATASET
    candidates = []
    for train_dir in MYDRIVE.rglob('train'):
        candidate = train_dir.parent
        if has_clean_layout(candidate):
            candidates.append(candidate)
    candidates = sorted({path.resolve() for path in candidates})
    if not candidates:
        raise FileNotFoundError('Stage2_Vietnam_Clean was not found in MyDrive')
    if len(candidates) > 1:
        raise RuntimeError(f'Ambiguous clean dataset candidates: {candidates}')
    return candidates[0]

DATASET_ROOT = find_clean_dataset()
print('Dataset:', DATASET_ROOT)

In [ ]:
def image_paths(folder):
    return sorted(path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

def count_images(root):
    return {split: {class_name: len(image_paths(root / split / class_name)) for class_name in CLASS_NAMES} for split in SPLITS}

actual_counts = count_images(DATASET_ROOT)
print(json.dumps(actual_counts, indent=2))
assert actual_counts == EXPECTED, f'Clean counts changed: {actual_counts}'
assert sum(actual_counts['train'].values()) == 1617
assert sum(actual_counts['valid'].values()) == 187
assert sum(actual_counts['test'].values()) == 178

# Test lock: existence/count only. No pixels are opened and no model call uses TEST_ROOT.
TEST_ROOT = DATASET_ROOT / 'test'
assert TEST_ROOT.is_dir()
TEST_IMAGE_COUNT = sum(actual_counts['test'].values())
assert TEST_IMAGE_COUNT == 178
TEST_PREDICTION_COUNT = 0
TEST_USED = False
print('TEST WILL NOT BE EVALUATED IN THIS NOTEBOOK.')

In [ ]:
MODEL_SOURCE = 'yolo26s-cls.pt'
assert Path(MODEL_SOURCE).name == 'yolo26s-cls.pt'
model = YOLO(MODEL_SOURCE)
assert model.task == 'classify', model.task

def normalized_names(names):
    if isinstance(names, dict):
        return {int(key): str(value) for key, value in names.items()}
    return {index: str(value) for index, value in enumerate(names)}

print('model.task =', model.task)
MODEL_NAMES = normalized_names(model.names)
assert MODEL_NAMES, 'Pretrained model.names is empty'
print('pretrained model.names sample:', dict(list(MODEL_NAMES.items())[:5]))
print('Class IDs will be resolved from the trained checkpoint by class name.')

In [ ]:
assert torch.cuda.is_available(), 'CUDA GPU is required; stopping before training.'
GPU_NAME = torch.cuda.get_device_name(0)
assert not RUN_DIR.exists(), f'Experiment folder already exists: {RUN_DIR}'

ultralytics_dataset_info = check_cls_dataset(DATASET_ROOT, split='val')
assert Path(ultralytics_dataset_info['train']).resolve() == (DATASET_ROOT / 'train').resolve()
assert Path(ultralytics_dataset_info['val']).resolve() == (DATASET_ROOT / 'valid').resolve(), ultralytics_dataset_info
assert Path(ultralytics_dataset_info['test']).resolve() == (DATASET_ROOT / 'test').resolve()
print('Ultralytics classification val split:', ultralytics_dataset_info['val'])
print('Ultralytics test path inspected for count only:', ultralytics_dataset_info['test'])

TRAIN_KWARGS = {
    'data': str(DATASET_ROOT),
    'epochs': 50,
    'imgsz': 224,
    'batch': 32,
    'device': 0,
    'workers': 4,
    'optimizer': 'AdamW',
    'lr0': 0.0005,
    'weight_decay': 0.0005,
    'patience': 10,
    'seed': 0,
    'deterministic': True,
    'pretrained': True,
    'amp': True,
    'hsv_h': 0.01,
    'hsv_s': 0.20,
    'hsv_v': 0.20,
    'degrees': 5.0,
    'translate': 0.05,
    'scale': 0.10,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'erasing': 0.0,
    'project': str(PROJECT_DIR),
    'name': RUN_NAME,
    'exist_ok': False,
    'plots': True,
    'save': True,
}
if 'auto_augment' in inspect.signature(model.train).parameters:
    TRAIN_KWARGS['auto_augment'] = None

print('Ultralytics version:', ultralytics.__version__)
print('GPU:', GPU_NAME)
print('model:', MODEL_SOURCE)
print('task:', model.task)
print('dataset:', DATASET_ROOT)
print('train:', actual_counts['train'])
print('valid:', actual_counts['valid'])
print('test count only:', TEST_IMAGE_COUNT)
print('train kwargs:', {key: value for key, value in TRAIN_KWARGS.items() if key not in {'data', 'project'}})
print('TEST WILL NOT BE EVALUATED.')

In [ ]:
# TRAINING CELL — intentionally not executed while this notebook is being created.
results = model.train(**TRAIN_KWARGS)

In [ ]:
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
assert BEST_PT.exists(), BEST_PT
best_model = YOLO(str(BEST_PT))
assert best_model.task == 'classify', best_model.task
BEST_NAMES = normalized_names(best_model.names)
assert set(BEST_NAMES.values()) == set(CLASS_NAMES), BEST_NAMES
NAME_TO_ID = {name: class_id for class_id, name in BEST_NAMES.items()}
HELMET_ID = NAME_TO_ID['helmet']
NO_HELMET_ID = NAME_TO_ID['no_helmet']
assert {HELMET_ID, NO_HELMET_ID} == {0, 1}, BEST_NAMES
CLASS_MAPPING = {str(class_id): name for class_id, name in BEST_NAMES.items()}
(RUN_DIR / 'class_mapping.json').write_text(json.dumps(CLASS_MAPPING, indent=2), encoding='utf-8')
print('best checkpoint:', BEST_PT)
print('best model task:', best_model.task)
print('best model names:', BEST_NAMES)

In [ ]:
RESULTS_CSV = RUN_DIR / 'results.csv'
history = pd.read_csv(RESULTS_CSV)
print('history columns:', list(history.columns))

def first_column(frame, candidates):
    return next((column for column in candidates if column in frame.columns), None)

top1_column = first_column(history, ['metrics/accuracy_top1', 'metrics/accuracy_top1(B)'])
top5_column = first_column(history, ['metrics/accuracy_top5', 'metrics/accuracy_top5(B)'])
val_loss_column = first_column(history, ['val/loss', 'val/loss_cls'])
train_loss_column = first_column(history, ['train/loss', 'train/loss_cls'])
best_top1_epoch = int(history.loc[history[top1_column].idxmax(), 'epoch']) if top1_column else None
best_top5_epoch = int(history.loc[history[top5_column].idxmax(), 'epoch']) if top5_column else None
lowest_val_loss_epoch = int(history.loc[history[val_loss_column].idxmin(), 'epoch']) if val_loss_column else None
print('best top1 epoch:', best_top1_epoch)
print('best top5 epoch:', best_top5_epoch, '(reported only if available)')
print('lowest validation-loss epoch:', lowest_val_loss_epoch)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if train_loss_column:
    axes[0].plot(history['epoch'], history[train_loss_column], label=train_loss_column)
if val_loss_column:
    axes[0].plot(history['epoch'], history[val_loss_column], label=val_loss_column)
axes[0].set_title('Observed loss curves')
axes[0].set_xlabel('epoch')
axes[0].legend()
if top1_column:
    axes[1].plot(history['epoch'], history[top1_column], label=top1_column)
if top5_column:
    axes[1].plot(history['epoch'], history[top5_column], label=top5_column)
axes[1].set_title('Observed top-1/top-5 accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_history.png', dpi=160)
plt.show()
if train_loss_column and val_loss_column:
    best_val = history[val_loss_column].min()
    final_val = history[val_loss_column].iloc[-1]
    best_train = history[train_loss_column].min()
    final_train = history[train_loss_column].iloc[-1]
    if final_val > best_val * 1.10 and final_train <= best_train * 1.10:
        curve_observation = 'possible overfit'
    elif final_train > best_train * 1.10 and final_val > best_val * 1.10:
        curve_observation = 'possible underfit/under-training'
    else:
        curve_observation = 'stable/uncertain; observed curves only'
    print('curve observation:', curve_observation)

In [ ]:
VALID_ROOT = DATASET_ROOT / 'valid'
valid_helmet_images = image_paths(VALID_ROOT / 'helmet')
valid_nohelmet_images = image_paths(VALID_ROOT / 'no_helmet')
valid_image_paths = valid_helmet_images + valid_nohelmet_images
assert len(valid_helmet_images) == 115
assert len(valid_nohelmet_images) == 72
assert len(valid_image_paths) == 187

validation_rows = []
for path in valid_helmet_images:
    validation_rows.append({'image': str(path), 'actual_class': 'helmet', 'actual_id': HELMET_ID})
for path in valid_nohelmet_images:
    validation_rows.append({'image': str(path), 'actual_class': 'no_helmet', 'actual_id': NO_HELMET_ID})
validation_df = pd.DataFrame(validation_rows)
print(validation_df['actual_class'].value_counts().to_dict())
print('TEST predictions so far:', TEST_PREDICTION_COUNT)

In [ ]:
# Validation-only inference. TEST_ROOT is never passed to predict/val in this notebook.
validation_results = best_model.predict(
    source=[str(path) for path in valid_image_paths],
    imgsz=224,
    batch=32,
    device=0,
    verbose=False,
    save=False,
)
assert len(validation_results) == 187

prediction_rows = []
for row, result in zip(validation_rows, validation_results):
    probability = result.probs.data.detach().float().cpu().numpy()
    prob_helmet = float(probability[HELMET_ID])
    prob_nohelmet = float(probability[NO_HELMET_ID])
    predicted_at_05 = 'no_helmet' if prob_nohelmet >= 0.50 else 'helmet'
    prediction_rows.append({
        'image': row['image'],
        'actual_class': row['actual_class'],
        'prob_helmet': prob_helmet,
        'prob_nohelmet': prob_nohelmet,
        'predicted_at_05': predicted_at_05,
    })

predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
assert TEST_PREDICTION_COUNT == 0
print('validation predictions:', len(predictions_df))
print('test predictions:', TEST_PREDICTION_COUNT)

In [ ]:
CLASS_ORDER = [HELMET_ID, NO_HELMET_ID]
CLASS_LABELS = ['helmet', 'no_helmet']
POSITION = {class_id: index for index, class_id in enumerate(CLASS_ORDER)}

def metric_bundle(dataframe, threshold):
    y_true = [int(value) for value in dataframe['actual_class'].map(NAME_TO_ID)]
    y_pred = [NO_HELMET_ID if value >= threshold else HELMET_ID for value in dataframe['prob_nohelmet']]
    matrix = np.zeros((2, 2), dtype=int)
    for actual, predicted in zip(y_true, y_pred):
        matrix[POSITION[predicted], POSITION[actual]] += 1
    precisions, recalls, f1s = [], [], []
    for position in range(2):
        tp = matrix[position, position]
        fp = matrix[position, :].sum() - tp
        fn = matrix[:, position].sum() - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    no_position = POSITION[NO_HELMET_ID]
    nohelmet_total = int(matrix[:, no_position].sum())
    nohelmet_correct = int(matrix[no_position, no_position])
    return {
        'threshold': float(threshold),
        'matrix': matrix,
        'accuracy': float(np.mean(np.array(y_true) == np.array(y_pred))),
        'macro_precision': float(np.mean(precisions)),
        'macro_recall': float(np.mean(recalls)),
        'macro_f1': float(np.mean(f1s)),
        'helmet_precision': float(precisions[POSITION[HELMET_ID]]),
        'helmet_recall': float(recalls[POSITION[HELMET_ID]]),
        'helmet_f1': float(f1s[POSITION[HELMET_ID]]),
        'nohelmet_precision': float(precisions[no_position]),
        'nohelmet_recall': float(recalls[no_position]),
        'nohelmet_f1': float(f1s[no_position]),
        'nohelmet_correct': nohelmet_correct,
        'nohelmet_total': nohelmet_total,
        'nohelmet_diagonal': nohelmet_correct / nohelmet_total if nohelmet_total else 0.0,
        'false_helmet': int(matrix[POSITION[HELMET_ID], no_position]),
        'false_nohelmet': int(matrix[no_position, POSITION[HELMET_ID]]),
    }

metrics_050 = metric_bundle(predictions_df, 0.50)
assert metrics_050['nohelmet_total'] == 72
print('confusion matrix rows=predicted, columns=actual')
print(metrics_050['matrix'])
print('no_helmet @ 0.50:', metrics_050['nohelmet_correct'], '/', metrics_050['nohelmet_total'], metrics_050['nohelmet_diagonal'])

In [ ]:
threshold_rows = []
for threshold in np.round(np.arange(0.20, 0.801, 0.01), 2):
    metrics = metric_bundle(predictions_df, float(threshold))
    threshold_rows.append({key: value for key, value in metrics.items() if key != 'matrix'})
threshold_sweep_df = pd.DataFrame(threshold_rows)
threshold_sweep_df.to_csv(RUN_DIR / 'threshold_sweep.csv', index=False)

candidates = threshold_sweep_df[
    (threshold_sweep_df['nohelmet_diagonal'] > 0.90)
    & (threshold_sweep_df['nohelmet_precision'] >= 0.85)
].copy()
if len(candidates):
    selected_pool = candidates.copy()
    target_reached = True
else:
    selected_pool = threshold_sweep_df.copy()
    target_reached = False
selected_pool['_distance_from_05'] = (selected_pool['threshold'] - 0.50).abs()
selected_row = selected_pool.sort_values(
    ['nohelmet_f1', 'macro_f1', '_distance_from_05'],
    ascending=[False, False, True],
    kind='mergesort',
).iloc[0]

SELECTED_THRESHOLD = float(selected_row['threshold'])
selected_metrics = metric_bundle(predictions_df, SELECTED_THRESHOLD)
threshold_payload = {
    'model': 'yolo26s-cls.pt',
    'positive_class': 'no_helmet',
    'threshold': SELECTED_THRESHOLD,
    'selected_on': 'Stage2_Vietnam_Clean valid',
    'valid_nohelmet_count': 72,
    'test_used': False,
}
(RUN_DIR / 'stage2_threshold.json').write_text(json.dumps(threshold_payload, indent=2), encoding='utf-8')
print('selected threshold:', SELECTED_THRESHOLD)
print('target reached:', target_reached)
print('selected no_helmet:', selected_metrics['nohelmet_correct'], '/', selected_metrics['nohelmet_total'], selected_metrics['nohelmet_diagonal'])

In [ ]:
def save_confusion(matrix, path, normalized=False):
    values = matrix.astype(float)
    if normalized:
        values = values / np.where(values.sum(axis=0, keepdims=True) == 0, 1, values.sum(axis=0, keepdims=True))
    fig, axis = plt.subplots(figsize=(5, 4))
    image = axis.imshow(values, cmap='Blues', vmin=0, vmax=1 if normalized else None)
    axis.set_xticks(range(2), CLASS_LABELS)
    axis.set_yticks(range(2), CLASS_LABELS)
    axis.set_xlabel('Actual')
    axis.set_ylabel('Predicted')
    axis.set_title('Rows=Predicted, Columns=Actual')
    for row in range(2):
        for column in range(2):
            label = f'{values[row, column]:.3f}' if normalized else str(int(values[row, column]))
            axis.text(column, row, label, ha='center', va='center')
    fig.colorbar(image, ax=axis)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

save_confusion(metrics_050['matrix'], RUN_DIR / 'confusion_matrix_val_050.png')
save_confusion(metrics_050['matrix'], RUN_DIR / 'confusion_matrix_val_050_normalized.png', normalized=True)
save_confusion(selected_metrics['matrix'], RUN_DIR / 'confusion_matrix_val_selected.png')
save_confusion(selected_metrics['matrix'], RUN_DIR / 'confusion_matrix_val_selected_normalized.png', normalized=True)

y_true_binary = (predictions_df['actual_class'] == 'no_helmet').astype(int).to_numpy()
scores = predictions_df['prob_nohelmet'].to_numpy()
precision_curve, recall_curve, _ = precision_recall_curve(y_true_binary, scores)
fpr, tpr, _ = roc_curve(y_true_binary, scores)
PR_AUC = float(average_precision_score(y_true_binary, scores))
ROC_AUC = float(roc_auc_score(y_true_binary, scores))
fig, axis = plt.subplots(figsize=(5, 4))
axis.plot(recall_curve, precision_curve, label=f'AP={PR_AUC:.4f}')
axis.set_xlabel('Recall: no_helmet')
axis.set_ylabel('Precision: no_helmet')
axis.set_title('Validation PR curve')
axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'nohelmet_pr_curve.png', dpi=160)
plt.close(fig)
fig, axis = plt.subplots(figsize=(5, 4))
axis.plot(fpr, tpr, label=f'ROC-AUC={ROC_AUC:.4f}')
axis.plot([0, 1], [0, 1], '--', color='gray')
axis.set_xlabel('False positive rate')
axis.set_ylabel('True positive rate')
axis.set_title('Validation ROC curve')
axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'nohelmet_roc_curve.png', dpi=160)
plt.close(fig)
print('PR-AUC:', PR_AUC, 'ROC-AUC:', ROC_AUC)

In [ ]:
selected_predictions = predictions_df.copy()
selected_predictions['predicted'] = np.where(
    selected_predictions['prob_nohelmet'] >= SELECTED_THRESHOLD, 'no_helmet', 'helmet'
)
selected_predictions['selected_threshold'] = SELECTED_THRESHOLD
errors = selected_predictions[selected_predictions['actual_class'] != selected_predictions['predicted']].copy()
errors['error_type'] = np.where(
    (errors['actual_class'] == 'no_helmet') & (errors['predicted'] == 'helmet'),
    'NOHELMET_TO_HELMET',
    'HELMET_TO_NOHELMET',
)
errors['actual'] = errors['actual_class']
errors = errors[['image', 'actual', 'predicted', 'prob_helmet', 'prob_nohelmet', 'selected_threshold', 'error_type']]
errors.to_csv(RUN_DIR / 'validation_errors.csv', index=False)
for error_type in ('nohelmet_to_helmet', 'helmet_to_nohelmet'):
    (RUN_DIR / 'validation_errors' / error_type).mkdir(parents=True, exist_ok=True)
for error_type, group in errors.groupby('error_type'):
    destination_dir = RUN_DIR / 'validation_errors' / error_type.lower()
    destination_dir.mkdir(parents=True, exist_ok=True)
    for source in group['image']:
        destination = destination_dir / Path(source).name
        if destination.exists():
            destination = destination_dir / f'{Path(source).stem}__duplicate{Path(source).suffix}'
        shutil.copy2(source, destination)
print('validation errors:', len(errors))

In [ ]:
benchmark_paths = valid_image_paths[:50]
assert len(benchmark_paths) <= 50
_ = best_model.predict(source=[str(path) for path in benchmark_paths[:4]], imgsz=224, batch=4, device=0, verbose=False, save=False)
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
benchmark_results = best_model.predict(
    source=[str(path) for path in benchmark_paths],
    imgsz=224,
    batch=32,
    device=0,
    verbose=False,
    save=False,
)
inference_ms = [float(result.speed['inference']) for result in benchmark_results if 'inference' in result.speed]
assert inference_ms, 'No inference timing was returned'
MEAN_INFERENCE_MS = float(np.mean(inference_ms))
MEDIAN_INFERENCE_MS = float(np.median(inference_ms))
FPS = 1000.0 / MEAN_INFERENCE_MS if MEAN_INFERENCE_MS else 0.0
GPU_PEAK_VRAM_MB = float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else None
print('mean inference ms/image:', MEAN_INFERENCE_MS)
print('median inference ms/image:', MEDIAN_INFERENCE_MS)
print('FPS:', FPS)
print('GPU peak VRAM MB:', GPU_PEAK_VRAM_MB)

In [ ]:
summary_row = {
    'Model': 'YOLO26s Classification',
    'Task': 'classify',
    'Dataset': 'Stage2_Vietnam_Clean',
    'imgsz': 224,
    'epochs_requested': 50,
    'batch': 32,
    'optimizer': 'AdamW',
    'lr0': 0.0005,
    'Checkpoint': str(BEST_PT),
    'Threshold': SELECTED_THRESHOLD,
    'Accuracy': selected_metrics['accuracy'],
    'MacroPrecision': selected_metrics['macro_precision'],
    'MacroRecall': selected_metrics['macro_recall'],
    'MacroF1': selected_metrics['macro_f1'],
    'HelmetPrecision': selected_metrics['helmet_precision'],
    'HelmetRecall': selected_metrics['helmet_recall'],
    'HelmetF1': selected_metrics['helmet_f1'],
    'NoHelmetPrecision': selected_metrics['nohelmet_precision'],
    'NoHelmetRecall': selected_metrics['nohelmet_recall'],
    'NoHelmetF1': selected_metrics['nohelmet_f1'],
    'NoHelmetCorrect': selected_metrics['nohelmet_correct'],
    'NoHelmetTotal': selected_metrics['nohelmet_total'],
    'NoHelmetDiagonal': selected_metrics['nohelmet_diagonal'],
    'PRAUC': PR_AUC,
    'ROCAUC': ROC_AUC,
    'Target90Pass': target_reached,
    'TestUsed': False,
}
pd.DataFrame([summary_row]).to_csv(RUN_DIR / 'stage2_yolo26s_val_summary.csv', index=False)

summary_lines = [
    'MODEL', 'YOLO26s Classification', '(yolo26s-cls.pt)', '',
    'DATASET', 'Stage2_Vietnam_Clean', '',
    'TRAIN', 'helmet 852', 'no_helmet 765', '',
    'VALID', 'helmet 115', 'no_helmet 72', '',
    'TEST', 'helmet 107', 'no_helmet 71', '',
    'TEST USED: NO', '',
    'THRESHOLD @ 0.50', f"NoHelmet {metrics_050['nohelmet_correct']} / {metrics_050['nohelmet_total']} = {metrics_050['nohelmet_diagonal']:.4f}", '',
    'SELECTED THRESHOLD', str(SELECTED_THRESHOLD), f"NoHelmet {selected_metrics['nohelmet_correct']} / {selected_metrics['nohelmet_total']} = {selected_metrics['nohelmet_diagonal']:.4f}",
    'TARGET >90', 'PASS' if target_reached else 'FAIL — TARGET NOT REACHED', '',
    'Macro F1', f"{selected_metrics['macro_f1']:.4f}",
    'PR-AUC', f"{PR_AUC:.4f}", 'ROC-AUC', f"{ROC_AUC:.4f}",
    'Inference ms/image mean', f"{MEAN_INFERENCE_MS:.4f}", 'median', f"{MEDIAN_INFERENCE_MS:.4f}", 'FPS', f"{FPS:.4f}",
    'TEST EVALUATED: NO', 'MODEL TRAINING: YES',
]
(RUN_DIR / 'stage2_yolo26s_val_summary.txt').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print('summary saved to:', RUN_DIR)

In [ ]:
assert ultralytics.__version__ == '8.4.155'
assert Path(MODEL_SOURCE).name == 'yolo26s-cls.pt'
assert best_model.task == 'classify'
assert sum(actual_counts['train'].values()) == 1617
assert sum(actual_counts['valid'].values()) == 187
assert sum(actual_counts['test'].values()) == 178
assert TEST_PREDICTION_COUNT == 0
assert TEST_USED is False
assert BEST_PT.exists()
assert selected_metrics['nohelmet_total'] == 72
assert (RUN_DIR / 'stage2_threshold.json').exists()
assert (RUN_DIR / 'class_mapping.json').exists()

print('========================================')
print('STAGE2 YOLO26S CLASSIFICATION COMPLETE')
print('========================================')
print('Model: YOLO26s Classification (yolo26s-cls.pt)')
print('Dataset: Stage2_Vietnam_Clean')
print('TRAIN: 1617 | VALID: 187 | TEST: 178')
print('TEST USED: NO')
print('Input: 224 | Epochs requested: 50 | Batch: 32')
print('Best checkpoint:', BEST_PT)
print('Selected threshold:', SELECTED_THRESHOLD)
print('NoHelmet:', selected_metrics['nohelmet_precision'], selected_metrics['nohelmet_recall'], selected_metrics['nohelmet_f1'])
print('Correct:', selected_metrics['nohelmet_correct'], '/', selected_metrics['nohelmet_total'])
print('Diagonal:', f"{selected_metrics['nohelmet_diagonal'] * 100:.2f}%")
print('Required for >90%: 65 / 72')
print('TARGET:', 'PASS' if target_reached else 'FAIL')
print('Macro F1:', selected_metrics['macro_f1'])
print('PR-AUC:', PR_AUC, '| ROC-AUC:', ROC_AUC)
print('Inference:', MEAN_INFERENCE_MS, 'ms/image')
print('TEST EVALUATED: NO')
print('========================================')